<a href="https://colab.research.google.com/github/msiplab/VieWork/blob/master/code/python/vie_sec10_restoration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> <a href="https://matlab.mathworks.com/open/github/v1?repo=msiplab/VieWork&file=code/matlab/vie_sec10_restoration.m&project=VieWork.prj" target="_parent"><img src="https://www.mathworks.com/images/responsive/global/open-in-matlab-online.svg" alt="Open in MATLAB Online"/></a>

# 第10回 画像復元
村松正吾　「画像情報工学」

動作確認：Python 3.12（Google Colab），NumPy，SciPy，scikit-image，Matplotlib，PyWavelets，Pillow

MATLAB のライブスクリプト `code/matlab/vie_sec10_restoration.m` を Python に移植したもの。第10回のスライド（vie2026-10）で使う図と数値を計算する。DCT の問題点（ブロックノイズ・モスキートノイズ）と離散ウェーブレット変換（DWT），観測モデル $\mathbf{v}=\mathbf{H}\mathbf{x}+\mathbf{w}$，最小二乗法と勾配降下法（GD 法），スパースモデリング（合成モデル $\mathbf{x}=\mathbf{D}\mathbf{s}$，1-ノルム正則化），ISTA／FISTA によるボケ＋ノイズ除去を扱う。記号は教科書に合わせる（ステップサイズ $\eta$，閾値 $\tau$，評価関数 $\mathfrak{J}$）。

スライドの図と数値は MATLAB 版で作っている。新潟大学の学生は[大学の包括ライセンス](https://jp.mathworks.com/academia/tah-portal/niigata-university-31576302.html)で MATLAB を使えるので，MATLAB 版もぜひ動かしてほしい。上の「Open in MATLAB Online」のボタンを押すと，ブラウザだけで MATLAB Online が開き，VieWork が自分の MATLAB Drive に複製されて，このノートブックに対応するライブスクリプトが開く（インストールは不要）。さらに [MATLAB Connector](https://jp.mathworks.com/products/matlab-drive.html) を手元の PC に入れると，MATLAB Drive のファイルが PC のフォルダと自動で同期されるので，MATLAB Online で作業したファイルを手元の MATLAB でもそのまま開ける。

画像処理には scikit-image，SciPy，NumPy，PyWavelets，Pillow を使う。教科書の例題の数値はスライドと一致することを `assert` で確かめている。画像から計算する値は，ライブラリの実装が MATLAB と異なるため，スライドの値とわずかに異なることがある。

## 準備
共通ヘルパー `vie.py`（サンプル画像の取得，日本語フォントと配色の設定）を読み込む。Colab では GitHub から取得する。

In [ ]:
import os
import io
import warnings
import urllib.request

if not os.path.exists("vie.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/msiplab/VieWork/master/code/python/vie.py", "vie.py")
try:
    import pywt
except ImportError:
    %pip install -q PyWavelets
    import pywt

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import dctn
from PIL import Image
from skimage.filters import gaussian
from skimage.metrics import peak_signal_noise_ratio
from skimage.util import img_as_float, img_as_ubyte
import vie

# 図の配色はスライドと揃え，ロゴの 3 色（メインの緑，寒色系の青，暖色系の橙）と灰色を使う
cmain, ccool, cwarm, cgray = vie.CMAIN, vie.CCOOL, vie.CWARM, vie.CGRAY


def show(ax, X, title=None):
    """画像を軸なし・縦横比 1 で表示する（グレースケールは 0 を黒，最大値（255 か 1）を白）．"""
    vmax = 255 if X.dtype == np.uint8 else 1
    ax.imshow(X, cmap="gray", vmin=0, vmax=vmax, interpolation="nearest")
    ax.set_axis_off()
    if title is not None:
        ax.set_title(title)

PSNR（ピーク値は uint8 なら 255，実数なら 1），ガウス関数の PSF，画像を JPEG／JPEG2000 で圧縮して読み戻す関数を用意する（ファイルには書き出さず，メモリ上で符号化する）。

In [ ]:
def psnr(A, ref):
    """ref を原画像とする A の PSNR [dB]（ピーク値は uint8 なら 255，実数なら 1）．"""
    return peak_signal_noise_ratio(ref, A, data_range=255 if ref.dtype == np.uint8 else 1)


def gauss_psf(size, sigma):
    """大きさ size×size，標準偏差 sigma のガウス関数の PSF（総和 1）．"""
    x = np.arange(size) - (size - 1) / 2
    h = np.exp(-(x[:, None] ** 2 + x[None, :] ** 2) / (2 * sigma**2))
    return h / h.sum()


def encode(X, fmt, **kwargs):
    """uint8 の画像 X を Pillow で符号化し，(バイト数, 読み戻した uint8 の画像) を返す．"""
    buf = io.BytesIO()
    Image.fromarray(X).save(buf, fmt, **kwargs)
    nbytes = buf.tell()
    buf.seek(0)
    return nbytes, np.asarray(Image.open(buf))

## DCT の問題点：ブロックノイズとモスキートノイズ
低画質（品質 5）の JPEG で圧縮すると，$8\times8$ ブロックの境界が見える（ブロックノイズ）。

画像は教科書のサンプル画像 msipimg01.tif（海岸）を $256\times256$ 画素のグレースケールに縮小して使う（共通関数 `vie.msipimg`）。なめらかな空ではブロックの階段状の境界が，水平線と桟橋の輪郭の上下には縞状の揺らぎがはっきり見える（石像の顔 msipimg05，建物 msipimg04，路面 msipimg06 などと比べて選んだ）。

JPEG の符号化には Pillow（libjpeg）を使う。

In [ ]:
X1 = img_as_float(vie.msipimg(1, 256, "gray"))   # 海岸（256×256）
_, Xj = encode(img_as_ubyte(X1), "JPEG", quality=5)
Xj = img_as_float(Xj)
psnrJ = psnr(Xj, X1)
print(f"psnrJ = {psnrJ:.4f} dB")

モスキートノイズは，平坦な面に鋭いエッジがある画像で見やすい（写真では細かな質感に紛れる）。msipimg04（建物）を 2 値化した人工的な画像（平坦な 2 階調と鋭い輪郭）を中程度の品質 20 で圧縮し，アーチの頂部を拡大する。平坦な面にはブロック境界がほとんど出ず，輪郭に沿って波打ち（モスキートノイズ）が現れる。

In [ ]:
X4 = img_as_float(vie.msipimg(4, 256, "gray"))
B4 = 0.2 + 0.6 * (gaussian(X4, sigma=1.5) > 0.45)  # 2 値化した人工画像
_, B4j = encode(img_as_ubyte(B4), "JPEG", quality=20)
B4j = img_as_float(B4j)

上段は海岸の画像（左：原画像，中：品質 5 の JPEG，右：空の部分 $64\times64$ 画素の拡大），下段は 2 値化した建物の画像（左：原画像，中：品質 20 の JPEG，右：中央のアーチの頂部 $48\times48$ 画素の拡大）。

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(10, 7), layout="constrained")
show(axs[0, 0], X1, "原画像")
show(axs[0, 1], Xj, f"JPEG（品質 5，{psnrJ:.1f} dB）")
axs[0, 1].add_patch(plt.Rectangle((127.5, -0.5), 64, 64, fill=False, ec=cwarm, lw=1.5))
show(axs[0, 2], Xj[0:64, 128:192], "空の部分（ブロックノイズ）")
show(axs[1, 0], B4, "2 値化した人工画像")
show(axs[1, 1], B4j, "JPEG（品質 20）")
axs[1, 1].add_patch(plt.Rectangle((94.5, 23.5), 48, 48, fill=False, ec=cwarm, lw=1.5))
show(axs[1, 2], B4j[24:72, 95:143], "アーチの頂部（モスキートノイズ）")
plt.show()

## JPEG と JPEG2000（同程度のファイルサイズ）
msipimg03.tif（マカロン，カラー $512\times512$ 画素）を約 12 kB になるよう JPEG と JPEG2000 で圧縮して比べる。なめらかなマカロンの表面で JPEG のブロックノイズが目立つ（花束 msipimg02，石像の顔 msipimg05 と比べて選んだ）。

JPEG2000 の符号化には Pillow（OpenJPEG）の非可逆 9/7 ウェーブレットと色変換（MCT）を使い，JPEG と同じ圧縮率を指定する。

In [ ]:
Xc = vie.msipimg(3)                             # マカロン（カラー，512×512）
target = 12e3                                   # 目標ファイルサイズ [bytes]
q, sz = 1, 0
while sz < target and q < 100:                  # 品質を上げながら目標サイズを超える手前を探す
    q += 1
    sz, _ = encode(Xc, "JPEG", quality=q)
q -= 1
szJ, Yj = encode(Xc, "JPEG", quality=q)
cr = Xc.size / szJ                              # JPEG と同じ圧縮率
szJ2, Yj2 = encode(Xc, "JPEG2000", quality_mode="rates", quality_layers=[cr], irreversible=True, mct=1)
sizes = [szJ, szJ2]
psnrs = [psnr(Yj, Xc), psnr(Yj2, Xc)]
print("品質 q =", q)
print("sizes =", sizes)
print("psnrs =", np.round(psnrs, 4))
print(f"vie-10-size-jpg: {szJ / 1e3:.1f}，vie-10-size-jp2: {szJ2 / 1e3:.1f} [kB]")
print(f"vie-10-psnr-jpg: {psnrs[0]:.1f}，vie-10-psnr-jp2: {psnrs[1]:.1f} [dB]")

中央の $256\times256$ 画素を並べる。

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9, 4.8), layout="constrained")
show(axs[0], Yj[128:384, 128:384], f"JPEG（{szJ / 1e3:.1f} kB，{psnrs[0]:.1f} dB）")
show(axs[1], Yj2[128:384, 128:384], f"JPEG2000（{szJ2 / 1e3:.1f} kB，{psnrs[1]:.1f} dB）")
plt.show()

## 2 次元 DWT（3 レベル）と DCT の周波数配置
9/7 DWT（PyWavelets の `bior4.4`）を 3 レベル施し，係数を周波数配置で並べる（多重解像度表現）。画像は第9回の vie-09-bdct と同じ msipimg04.tif（石造りの建物，256×256 のグレースケール）とし，第12回のスライドで並べたときに対応が分かるようにする。暗い空が平坦なので，係数の疎な様子が見やすい。

In [ ]:
Xs = img_as_float(vie.msipimg(4, 256, "gray"))   # 石造りの建物（256×256）
A1, (H1, V1, D1) = pywt.dwt2(Xs, "bior4.4", mode="periodization")
A2, (H2, V2, D2) = pywt.dwt2(A1, "bior4.4", mode="periodization")
A3, (H3, V3, D3) = pywt.dwt2(A2, "bior4.4", mode="periodization")


def sc(c):
    """係数の絶対値を見やすく表示"""
    return np.minimum(np.abs(c) * 4, 1)


L3 = np.block([[A3 / A3.max(), sc(H3)], [sc(V3), sc(D3)]])
L2 = np.block([[L3, sc(H2)], [sc(V2), sc(D2)]])
Wdwt = np.block([[L2, sc(H1)], [sc(V1), sc(D1)]])

比較のため，8×8 ブロック DCT の係数を周波数ごとに集めて並べ替える（各周波数 $(u,v)$ の係数が一枚の縮小画像になる）。左が 3 レベル DWT，右が並べ替えたブロック DCT の係数である。

In [ ]:
nb = Xs.shape[0] // 8
Yb = dctn(Xs.reshape(nb, 8, nb, 8), axes=(1, 3), norm="ortho").reshape(Xs.shape)   # ブロックごとの DCT
Wdct = np.zeros_like(Xs)
for u in range(8):
    for v in range(8):
        sub = Yb[u::8, v::8]
        sub = sub / sub.max() if (u == 0 and v == 0) else sc(sub / 2)
        Wdct[u * nb:(u + 1) * nb, v * nb:(v + 1) * nb] = sub
fig, axs = plt.subplots(1, 2, figsize=(9, 4.8), layout="constrained")
show(axs[0], Wdwt, "3 レベル 9/7 DWT")
show(axs[1], Wdct, "8×8 ブロック DCT（周波数ごとに並べ替え）")
plt.show()

## 3 段 9/7 DWT の合成 FB の周波数振幅応答
参考資料 図 6.12(b)（3 段ハール DWT）と同じ見せ方で，9/7 DWT（`bior4.4`）の 3 段 DWT と等価な不等分割 FB の合成フィルタの周波数振幅応答を描く。等価フィルタはノーブル恒等式から $F_{3,0}(z)=F_0(z^4)F_0(z^2)F_0(z)$，$F_{3,1}(z)=F_1(z^4)F_0(z^2)F_0(z)$，$F_{2,1}(z)=F_1(z^2)F_0(z)$，$F_{1,1}(z)=F_1(z)$。

In [ ]:
wav = pywt.Wavelet("bior4.4")
f0, f1 = np.array(wav.rec_lo), np.array(wav.rec_hi)   # 合成フィルタ F_0(z)，F_1(z)


def up(f, m):
    """z -> z^m（アップサンプル）"""
    g = np.zeros(len(f) * m)
    g[::m] = f
    return g


F30 = np.convolve(np.convolve(up(f0, 4), up(f0, 2)), f0)
F31 = np.convolve(np.convolve(up(f1, 4), up(f0, 2)), f0)
F21 = np.convolve(up(f1, 2), f0)
F11 = f1
Nw = 1024
w = np.linspace(0, np.pi, Nw)
fig, ax = plt.subplots(figsize=(4.5, 4.2), layout="constrained")
for Fi, sty, lab in zip((F30, F31, F21, F11), ("-", "--", ":", "-."),
                        ("$F_{3,0}(z)$", "$F_{3,1}(z)$", "$F_{2,1}(z)$", "$F_{1,1}(z)$")):
    _, Hw = signal.freqz(Fi, 1, worN=w)
    ax.plot(w, 20 * np.log10(np.abs(Hw) + np.finfo(float).eps), sty, color="k", lw=1.2, label=lab)
ax.set(xlim=(0, np.pi), ylim=(-30, 10), xticks=[0, np.pi / 2, np.pi],
       xticklabels=["$0$", r"$\pi/2$", r"$\pi$"], xlabel=r"$\omega$",
       ylabel=r"$20\log_{10}|F_{j,p}(\mathrm{e}^{\mathrm{j}\omega})|$ [dB]")
ax.grid(True)
ax.legend(loc="lower right")
plt.show()

## 時空間－周波数のタイル：ブロック DCT と 9/7 DWT（参考資料 6.4.2 項）
横軸を位置 $n$，縦軸を周波数 $\omega$ とする平面のタイルで，等分割 FB（8 点ブロック DCT）とオクターブ分割 FB（3 段 9/7 DWT）の時空間分解能を比べる。各帯域には，隣り合う二つの基底（合成フィルタのインパルス応答）をストライドだけずらして 2 色で重ねる。DCT の基底はブロック内に収まり重ならない（ストライド 8）。DWT の基底はタイルからはみ出して重なり，ストライドは高域ほど小さい（2，4，8）。

In [ ]:
Nt = 32                                          # 位置の範囲 0..Nt-1
C8 = dctn(np.eye(8), axes=(0,), norm="ortho").T  # 8 点 DCT の基底ベクトル（列）
bands97 = [F11, F21, F31, F30]                   # 高域から：F_{1,1}, F_{2,1}, F_{3,1}, F_{3,0}
lo97 = [np.pi / 2, np.pi / 4, np.pi / 8, 0]
hi97 = [np.pi, np.pi / 2, np.pi / 4, np.pi / 8]
st97 = [2, 4, 8, 8]
cols = (ccool, cwarm)
fig, axs = plt.subplots(1, 2, figsize=(10, 4.4), layout="constrained")
for panel, ax in enumerate(axs):
    if panel == 0:                               # ブロック DCT：8 帯域 × 幅 8
        lo = np.arange(8) * np.pi / 8
        hi = np.arange(1, 9) * np.pi / 8
        st = [8] * 8
    else:
        lo, hi, st = lo97, hi97, st97
    for b in range(len(lo)):
        for x0 in range(0, Nt, st[b]):           # タイル
            ax.add_patch(plt.Rectangle((x0 - 0.5, lo[b]), st[b], hi[b] - lo[b],
                                       ec=(0.7, 0.7, 0.7), fc=(0.93, 0.97, 0.95), lw=0.8))
        yc = (lo[b] + hi[b]) / 2
        amp = 0.42 * (hi[b] - lo[b])
        if panel == 0:
            g = C8[:, b]
            n0 = [8, 16]                         # 隣り合う 2 ブロック
        else:
            g = bands97[b]
            c = int(np.floor((Nt - len(g)) / 2 + 0.5))   # 四捨五入
            n0 = [c, c + st[b]]
        g = g / np.max(np.abs(g))
        if panel == 1 and g.max() < 1:
            g = -g                               # 合成高域フィルタは中央が負．ψ と同じ向き（中央が正）にそろえる
        if np.all(np.abs(g - g[0]) < 1e-12):
            g = 0.4 * g                          # 直流の基底は帯域の中ほどに描く
        for k in range(2):
            nn = n0[k] + np.arange(len(g))
            keep = (nn >= 0) & (nn <= Nt - 1)    # 表示範囲の標本だけ
            ax.plot(nn[keep], yc + amp * g[keep], "-", color=cols[k], lw=0.8)
            ax.plot(nn[keep], yc + amp * g[keep], ".", color=cols[k], ms=4)   # 標本点
        ax.text(Nt + 0.3, yc, f"{st[b]}", fontsize=8, ha="left", va="center")
    ax.set(xlim=(-0.5, Nt - 0.5), ylim=(0, np.pi), xticks=np.arange(0, Nt + 1, 8),
           yticks=[0, np.pi / 8, np.pi / 4, np.pi / 2, np.pi],
           yticklabels=["$0$", r"$\pi/8$", r"$\pi/4$", r"$\pi/2$", r"$\pi$"])
    ax.set_axisbelow(False)
    ax.set_xlabel("位置 $n$")
    ax.set_ylabel(r"$\omega$", rotation=0, labelpad=8)
    ax.text(Nt + 3.0, np.pi / 2, "ストライド", fontsize=8, rotation=90, ha="center", va="center")
    ax.set_title(["ブロック DCT（等分割 FB）", "3 段 9/7 DWT（オクターブ分割 FB）"][panel])
plt.show()

## 9/7 DWT の基底画像
各サブバンドの中央の係数だけを 1 にして逆変換すると，そのサブバンドの基底画像が得られる。サイズが周波数に応じて異なり，互いに重なり合う。

In [ ]:
Nb = 64
with warnings.catch_warnings():                 # 64×64 に 3 レベルは深いという警告を抑える
    warnings.simplefilter("ignore")
    c0 = pywt.wavedec2(np.zeros((Nb, Nb)), "bior4.4", mode="symmetric", level=3)   # [A3, (H3,V3,D3), (H2,…), (H1,…)]
names = ["A3", "H3", "V3", "D3", "H2", "V2", "D2", "H1", "V1", "D1"]
tiles = []
for name in names:
    c = [c0[0].copy()] + [tuple(b.copy() for b in lv) for lv in c0[1:]]
    typ, lev = name[0], int(name[1])
    if typ == "A":
        blk = c[0]
    else:
        blk = c[3 - lev + 1]["HVD".index(typ)]  # c[1] がレベル 3
    blk[(blk.shape[0] + 1) // 2 - 1, (blk.shape[1] + 1) // 2 - 1] = 1   # 中央の係数
    b = pywt.waverec2(c, "bior4.4", mode="symmetric")[:Nb, :Nb]
    tiles.append(0.5 + b / (2 * np.max(np.abs(b))))
Mos = np.block([[np.hstack(tiles[0:5])], [np.hstack(tiles[5:10])]])
fig, ax = plt.subplots(figsize=(10, 4.2), layout="constrained")
show(ax, Mos, "上段：A3，H3，V3，D3，H2　下段：V2，D2，H1，V1，D1")
plt.show()

## 画像復元の概要：ボケ（点広がり関数）
以降の画像復元の例では，教科書のサンプル画像 msipimg05.tif（石像の顔）を $256\times256$ 画素のグレースケールに縮小した画像を原画像 $\mathbf{x}$ とする。顔の輪郭や石の細かな模様があり，ボケとその除去の効果が見やすい。

In [ ]:
Xg = img_as_float(vie.msipimg(5, 256, "gray"))   # 石像の顔（256×256）

焦点ぼけなどは点広がり関数（PSF）との畳み込みで表される。PSF の例：標準偏差 $\sigma_\mathrm{g}=2$ のガウス関数。周期境界（循環畳み込み）とし，測定行列 $\mathbf{H}$ の積と随伴 $\mathbf{H}^\top$ の積を周波数領域で計算する（$\mathbf{H}^\top$ は相関，周波数応答の複素共役）。

周期境界の畳み込みの周波数応答は，PSF を画像の大きさに零詰めし，中心が原点（左上）に来るよう巡回シフトしてから二次元 DFT すれば求まる。

In [ ]:
psf = gauss_psf(25, 2)
P = np.zeros(Xg.shape)
P[:psf.shape[0], :psf.shape[1]] = psf
P = np.roll(P, (-(psf.shape[0] // 2), -(psf.shape[1] // 2)), axis=(0, 1))
Hf = np.fft.fft2(P)                             # 周期境界（循環畳み込み）の周波数応答


def blur(x):
    """H x"""
    return np.real(np.fft.ifft2(Hf * np.fft.fft2(x)))


def blurT(x):
    """H^T x"""
    return np.real(np.fft.ifft2(np.conj(Hf) * np.fft.fft2(x)))


Xb = blur(Xg)
dimg = np.zeros((64, 64))
dimg[32, 32] = 1                                # インパルス
pimg = signal.convolve2d(dimg, psf, mode="same")

原画像，ボケた画像，インパルスとその応答（PSF）を並べる。

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(11, 3.2), layout="constrained")
show(axs[0], Xg, "原画像 $\\mathbf{x}$")
show(axs[1], Xb, "ボケた画像 $\\mathbf{Hx}$")
show(axs[2], dimg, "インパルス")
show(axs[3], pimg / pimg.max(), "PSF")
plt.show()

## 観測ノイズ（AWGN）
前回スライドの例：3 段階の明るさの正方形に平均 0，分散 $\sigma_\mathrm{w}^2=0.001$ の白色ガウスノイズを加える。ヒストグラムの各ピークが正規分布に広がる。

In [ ]:
rng = np.random.default_rng(0)
U = 0.25 * np.ones((64, 64))
U[12:52, 12:52] = 0.5
U[24:40, 24:40] = 0.75
Vn = U + np.sqrt(0.001) * rng.standard_normal(U.shape)
fig, axs = plt.subplots(1, 4, figsize=(12, 3), layout="constrained")
show(axs[0], U, "原画像")
show(axs[1], np.clip(Vn, 0, 1), "観測画像")
edges = np.arange(0, 1.0001, 0.01)
for ax, Z, ttl in zip(axs[2:], (U, Vn), ("原画像のヒストグラム", "観測画像のヒストグラム")):
    ax.hist(Z.ravel(), bins=edges, color=ccool)
    ax.set(xlim=(0, 1), title=ttl)
plt.show()

## ボケ＋ノイズの観測画像
教科書と同じ設定：ガウシアン $\sigma_\mathrm{g}=2$ のボケと，分散 $(10/255)^2$ の AWGN。ノイズは 4 倍して 0.5 を足して表示する。

In [ ]:
rng = np.random.default_rng(1)
sigw = 10 / 255
Wn = sigw * rng.standard_normal(Xg.shape)
V = blur(Xg) + Wn
psnrV = psnr(V, Xg)
print(f"psnrV = {psnrV:.4f} dB")
print(f"vie-10-psnr-obs: {psnrV:.2f}")
fig, axs = plt.subplots(1, 2, figsize=(8, 4.3), layout="constrained")
show(axs[0], np.clip(V, 0, 1), f"観測画像 $\\mathbf{{v}}$（{psnrV:.2f} dB）")
show(axs[1], np.clip(0.5 + Wn * 4, 0, 1), "ノイズ $\\mathbf{w}$")
plt.show()

## 勾配降下法（GD 法）の概念図
前回スライドの手描きの図（評価関数のグラフ上で反復が最小値解に近づく）を描き直す。図のための一次元の評価関数 $\mathfrak{J}(x)=\frac12\|\mathbf{v}-\mathbf{H}x\|_2^2$（$\mathbf{H}=(1,1,1)^\top$，$\mathbf{v}=(1.2,0.9,0.9)^\top$）に GD 法 $x^{(t+1)}=x^{(t)}-\eta\,\mathbf{H}^\top(\mathbf{H}x^{(t)}-\mathbf{v})$（$\eta=0.2$，$x^{(0)}=0$）を適用する。数値はスライドに載せない。

In [ ]:
Hm = np.array([1.0, 1.0, 1.0])
vm = np.array([1.2, 0.9, 0.9])
xls = (Hm @ vm) / (Hm @ Hm)                     # 最小値解（正規方程式の解）
print("xls =", xls)
etaLs, nStep = 0.2, 6


def Jls(x):
    """評価関数 J(x)"""
    return 0.5 * np.sum((vm - Hm * x) ** 2)


xgd = np.zeros(nStep + 1)                       # x^(0), x^(1), ..., x^(nStep)
for t in range(nStep):
    xgd[t + 1] = xgd[t] - etaLs * Hm @ (Hm * xgd[t] - vm)
Jgd1 = np.array([Jls(x) for x in xgd])          # 評価関数の値（単調に減少）
print("xgd  =", np.round(xgd, 4))
print("Jgd1 =", np.round(Jgd1, 4))
Jmin = Jls(xls)

反復の様子を評価関数のグラフ上に描く（前回スライドの手描きの図を踏襲）。

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 3), layout="constrained")
xx = np.linspace(-0.15, 1.25, 200)
ax.plot(xx, [Jls(x) for x in xx], color=cmain, lw=1.5)
nShow = 3                                       # 矢印で示す反復の数
ax.plot(xgd[:nShow + 1], Jgd1[:nShow + 1], "o", ms=5, color=cwarm)
ax.plot(xls, Jmin, "o", ms=5, color=ccool)
ax.text(xgd[0] + 0.05, Jgd1[0] + 0.10, "$x^{(0)}$")
ax.text(xgd[1] - 0.06, Jgd1[1] - 0.12, "$x^{(1)}$", ha="right")
ax.text(xls + 0.06, Jmin + 0.20, r"$\hat{x}$", color=ccool)
shrink = 0.12                                   # 両端を縮める割合
for t in range(nShow):                          # x^(t) から x^(t+1) への矢印
    p, q = np.array([xgd[t], Jgd1[t]]), np.array([xgd[t + 1], Jgd1[t + 1]])
    ax.annotate("", xy=p + (1 - shrink) * (q - p), xytext=p + shrink * (q - p),
                arrowprops=dict(arrowstyle="-|>", color=cwarm, lw=1, mutation_scale=8))
ax.set(xlim=(-0.15, 1.25), ylim=(0, 1.9), xticks=[0, 0.5, 1], xlabel="$x$")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## GD 法によるボケ除去（評価関数の減少）
2025 年度の演習（`gdstep`）にならい，ボケ＋ノイズの観測画像 $\mathbf{v}$ に GD 法を適用する。ガウシアンフィルタは係数が非負で総和 1 なので $\|\mathbf{H}\|_\mathrm{S}=\max|H[\boldsymbol{k}]|=1$ となり，$\eta=1\in(0,2)$ とする。

$\lambda=0$（LS 法）と $\lambda=10^{-2}$，$\mathbf{L}=\mathbf{I}$（リッジ正則化）を比べる。どちらも評価関数 $\mathfrak{J}(\mathbf{x}^{(t)})$ は単調に減少する。LS 法は途中で PSNR が最大になったあと低下する（ノイズの増幅．不良設定問題）。リッジ正則化では推定が安定する。

In [ ]:
etaGd, nGd, lams = 1, 300, [0, 1e-2]
Jimg = np.zeros((nGd + 1, len(lams)))           # 評価関数と PSNR の履歴
Pimg = np.zeros_like(Jimg)
for k, lam in enumerate(lams):
    x = np.zeros_like(V)                        # 初期値 x^(0) = 0
    for t in range(nGd + 1):
        if t > 0:
            g = blurT(blur(x) - V) + lam * x    # 勾配 (H^T H + λI)x - H^T v
            x = x - etaGd * g                   # 勾配降下
        Jimg[t, k] = 0.5 * np.sum((V - blur(x)) ** 2) + lam / 2 * np.sum(x**2)
        Pimg[t, k] = psnr(x, Xg)
print("評価関数は単調に減少:", bool(np.all(np.diff(Jimg, axis=0) <= 1e-9)))
tPeak = int(np.argmax(Pimg[:, 0]))
psnrPeak = Pimg[tPeak, 0]
psnrEnd = Pimg[-1, :]
print("tPeak =", tPeak, f"（PSNR {psnrPeak:.2f} dB）")
print("psnrEnd =", np.round(psnrEnd, 4))
print(f"vie-10-gd-etaimg: {etaGd:g}，vie-10-gd-lam: 10^{{{int(np.round(np.log10(lams[1])))}}}")

上は評価関数（対数目盛），下は PSNR の推移。橙が $\lambda=0$，緑が $\lambda=10^{-2}$，灰色の点線が観測画像の PSNR である。

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(5.5, 4.8), sharex=True, layout="constrained")
tt = np.arange(1, nGd + 1)
axs[0].semilogy(tt, Jimg[tt, 0], color=cwarm, lw=1.5, label=r"$\lambda=0$")
axs[0].semilogy(tt, Jimg[tt, 1], color=cmain, lw=1.5, label=r"$\lambda=10^{-2}$")
axs[0].set(xlim=(0, nGd), ylim=(30, 200), ylabel="評価関数")
axs[0].set_yticks([50, 100, 200], ["50", "100", "200"])
axs[0].yaxis.set_minor_formatter(plt.NullFormatter())
axs[0].grid(True, which="both")
axs[0].legend()
axs[1].plot(tt, Pimg[tt, 0], color=cwarm, lw=1.5)
axs[1].plot(tt, Pimg[tt, 1], color=cmain, lw=1.5)
axs[1].axhline(psnrV, ls=":", color=cgray, lw=1)
axs[1].plot(tPeak, psnrPeak, "o", ms=5, color=cwarm)
axs[1].text(tPeak + 8, psnrPeak + 0.4, f"$t$={tPeak}", color=cwarm)
axs[1].text(nGd - 5, psnrV + 0.2, "観測", color=cgray, ha="right")
axs[1].set(xlim=(0, nGd), ylim=(17, 21.5), ylabel="PSNR [dB]", xlabel="反復回数 $t$")
axs[1].grid(True)
plt.show()

## ノイズと信号変換：ハール変換の詳細成分
原画像の詳細成分はほとんどが 0 付近（疎）だが，ノイズを加えると全体に広がる（密）。絶対値が閾値 0.02 未満の係数の割合で比べる。

この節（と次の節のヒストグラム）の画像は msipimg04.tif（石造りの建物，256×256 のグレースケール）とする。暗い空などの平坦な領域が広く，ノイズの有無による疎・密の違いがはっきり出る（8 枚の中で疎な係数の割合が最も大きい。石像の顔 msipimg05 では差が小さい）。

In [ ]:
Xs = img_as_float(vie.msipimg(4, 256, "gray"))   # 石造りの建物（256×256）
_, (Hc, Vc, Dc) = pywt.dwt2(Xs, "haar")
Xn = Xs + 20 / 255 * rng.standard_normal(Xs.shape)
_, (Hn, Vn2, Dn) = pywt.dwt2(Xn, "haar")
thSp = 0.02
sp = [np.mean(np.abs(Hc) < thSp), np.mean(np.abs(Hn) < thSp)]
print("sp =", np.round(sp, 4))
print(f"vie-10-sp-th: {thSp:g}，vie-10-sp-clean: {100 * sp[0]:.0f}，vie-10-sp-noisy: {100 * sp[1]:.0f} [%]")

上段は原画像とその水平方向の詳細成分（絶対値を 4 倍して表示），下段はノイズを加えた画像とその詳細成分。

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(7, 7), layout="constrained")
show(axs[0, 0], Xs, "原画像")
show(axs[0, 1], np.minimum(np.abs(Hc) * 4, 1), f"詳細成分（{100 * sp[0]:.0f} % が閾値未満）")
show(axs[1, 0], np.clip(Xn, 0, 1), "ノイズを加えた画像")
show(axs[1, 1], np.minimum(np.abs(Hn) * 4, 1), f"詳細成分（{100 * sp[1]:.0f} % が閾値未満）")
plt.show()

## 画像データについての事前知識：詳細成分のヒストグラム
ラプラス分布 $\mathcal{L}(s;0,b)=(2b)^{-1}\exp(-|s|/b)$ の尺度 $b$ を平均絶対値（最尤推定）で求めて重ねる。

In [ ]:
dcoef = np.concatenate([Hc.ravel(), Vc.ravel(), Dc.ravel()])
b = np.mean(np.abs(dcoef))                      # ラプラス分布の尺度（平均絶対値）
print(f"b = {b:.4f}")
print(f"vie-10-lap-b: {b:.3f}")
ss = np.linspace(-1, 1, 801)
fig, ax = plt.subplots(figsize=(5, 3.5), layout="constrained")
ax.hist(dcoef, bins=np.linspace(-1, 1, 401), density=True, color=ccool, alpha=0.55, label="ハール詳細成分")
ax.plot(ss, np.exp(-np.abs(ss) / b) / (2 * b), color=cwarm, lw=1.5, label="ラプラス分布")
ax.set(xlim=(-0.2, 0.2), xlabel="$s$", ylabel="確率密度")
ax.legend(fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## 1-ノルム正則化：ラプラス分布とガウス分布
同じ分散 $2b^2$（$b=0.1$）のラプラス分布とガウス分布を比べる。ラプラス分布は零の近くで尖り，裾が重い。

In [ ]:
bL = 0.1
sgG = np.sqrt(2) * bL
fig, ax = plt.subplots(figsize=(5, 3.6), layout="constrained")
ax.plot(ss, np.exp(-np.abs(ss) / bL) / (2 * bL), color=cwarm, lw=1.5, label="ラプラス分布（尖る）")
ax.plot(ss, np.exp(-ss**2 / (2 * sgG**2)) / (np.sqrt(2 * np.pi) * sgG), "--", color=ccool, lw=1.5,
        label="ガウス分布（同じ分散）")
ax.grid(True)
ax.set(xlim=(-0.5, 0.5), ylim=(0, 6.5), xlabel="$s$", ylabel="$p(s)$")
ax.legend(fontsize=9, loc="upper right")
plt.show()

## ソフト閾値処理
$\phi_\mathrm{ST}(x;\tau)=\mathrm{sign}(x)\max(|x|-\tau,0)$（$\tau=1$）。恒等写像 $y=x$（点線）も描く。$|x|\leq\tau$ は 0 に，それ以外は $\tau$ だけ原点に近づく。

In [ ]:
def st(x, tau):
    """ソフト閾値処理"""
    return np.sign(x) * np.maximum(np.abs(x) - tau, 0)


xx = np.linspace(-3, 3, 601)
fig, ax = plt.subplots(figsize=(4.5, 3.4), layout="constrained")
ax.plot(xx, xx, ":", color=cgray, lw=1)
ax.plot(xx, st(xx, 1), color=cmain, lw=1.8)
ax.grid(True)
ax.set_aspect("equal")
ax.set(xlim=(-3, 3), ylim=(-2.2, 2.2), xticks=[-3, -1, 0, 1, 3], yticks=[-2, -1, 0, 1, 2],
       xlabel="$x$", ylabel=r"$\phi_\mathrm{ST}(x;\tau)$")
plt.show()

## FISTA によるボケ＋ノイズ除去
合成辞書 $\mathbf{D}$ に 3 段の非間引きハール DWT（パーセバルタイト枠）を用いる。周期境界の畳み込みとして周波数領域で実装する。各段のフィルタは $\frac12(1\pm z^{-2^{j-1}})$。

In [ ]:
J = 3
N1, N2 = Xg.shape
w1, w2 = np.meshgrid(2 * np.pi * np.arange(N1) / N1, 2 * np.pi * np.arange(N2) / N2, indexing="ij")
Fl = []                                         # 各サブバンドの周波数応答
Lp = np.ones((N1, N2))
for j in range(1, J + 1):
    m = 2 ** (j - 1)
    h0v, h1v = (1 + np.exp(-1j * m * w1)) / 2, (1 - np.exp(-1j * m * w1)) / 2
    h0h, h1h = (1 + np.exp(-1j * m * w2)) / 2, (1 - np.exp(-1j * m * w2)) / 2
    Fl += [Lp * h0v * h1h, Lp * h1v * h0h, Lp * h1v * h1h]
    Lp = Lp * h0v * h0h
Fl.append(Lp)                                   # 最後が低域成分
F = np.stack(Fl, axis=2)                        # 3 次元目がサブバンド
K = F.shape[2]
tight = np.max(np.abs(np.sum(np.abs(F) ** 2, axis=2) - 1))   # Σ|F_k|^2 = 1（パーセバルタイト枠）
print("K =", K)
print("tight =", tight)

例題「ステップサイズ条件」の数値確認。周期境界の畳み込み $\mathbf{H}$ のスペクトルノルムは周波数振幅応答の最大値 $\max_{\boldsymbol{k}}|H[\boldsymbol{k}]|$。$\mathbf{H}\mathbf{D}\mathbf{D}^\top\mathbf{H}^\top$ も巡回行列で，その周波数応答は $|H[\boldsymbol{k}]|^2\sum_p|F_p[\boldsymbol{k}]|^2=|H[\boldsymbol{k}]|^2$ なので $\|\mathbf{H}\mathbf{D}\|_\mathrm{S}^2=\|\mathbf{H}\|_\mathrm{S}^2$。ガウシアンフィルタは直流で最大値 1 をとる。

In [ ]:
normH = np.max(np.abs(Hf))                      # ||H||_S
normHD2 = np.max(np.abs(Hf) ** 2 * np.sum(np.abs(F) ** 2, axis=2))   # ||HD||_S^2
print(f"normH = {normH:.6f}，normHD2 = {normHD2:.6f}")
print(f"vie-10-normH: {normH:.3f}，vie-10-normHD2: {normHD2:.3f}")
assert f"{normH:.3f}" == "1.000" and f"{normHD2:.3f}" == "1.000"

教科書と同じ設定（$\lambda=10^{-3}$，$\eta=1$，反復 20 回．実験的に決めたもの）で FISTA を実行する。$\eta = 1 \leq 1/\|\mathbf{H}\mathbf{D}\|_\mathrm{S}^2$ である。低域成分以外の係数をソフト閾値処理する。

In [ ]:
def T(x):
    """分析 D^T（係数は N1×N2×K）"""
    return np.real(np.fft.ifft2(F * np.fft.fft2(x)[:, :, None], axes=(0, 1)))


def Dm(c):
    """合成 D"""
    return np.real(np.fft.ifft2(np.sum(np.conj(F) * np.fft.fft2(c, axes=(0, 1)), axis=2)))


def HD(c):
    return blur(Dm(c))


def HDt(r):
    return T(blurT(r))


lam, eta, nIter = 1e-3, 1, 20
s = np.zeros((N1, N2, K))
y = s.copy()
a = 1.0
for t in range(nIter):
    g = HDt(HD(y) - V)                          # 勾配 D^T H^T (HDy - v)
    snew = y - eta * g                          # 勾配降下
    snew[:, :, :K - 1] = st(snew[:, :, :K - 1], lam * eta)   # 低域成分以外をソフト閾値処理
    anew = (1 + np.sqrt(1 + 4 * a**2)) / 2
    y = snew + (a - 1) / anew * (snew - s)      # ネステロフの加速
    s, a = snew, anew
Xr = np.clip(Dm(s), 0, 1)
psnrR = psnr(Xr, Xg)
print(f"psnrR = {psnrR:.4f} dB")
print(f"vie-10-psnr-rest: {psnrR:.2f}，vie-10-lambda: {lam:g}，vie-10-niter: {nIter}")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(11, 4), layout="constrained")
show(axs[0], Xg, "原画像")
show(axs[1], np.clip(V, 0, 1), f"観測画像（{psnrV:.2f} dB）")
show(axs[2], Xr, f"FISTA（{psnrR:.2f} dB）")
plt.show()

## まとめ
- DCT のブロック処理はブロックノイズ・モスキートノイズを生む。DWT は重なりのある多重解像度の基底でこれを抑える
- 観測モデル $\mathbf{v}=\mathbf{H}\mathbf{x}+\mathbf{w}$ から原画像を推定するのが画像復元（逆問題）
- GD 法は評価関数を単調に減少させる。LS 法（$\lambda=0$）はノイズを増幅するので，正則化で推定を安定させる
- 自然画像の変換係数は疎（ラプラス分布）で，1-ノルム正則化はこの事前知識を反映する
- ISTA／FISTA は勾配降下とソフト閾値処理を繰り返して解を求める

© Copyright, Shogo MURAMATSU, All rights reserved.